# Klyvaron LoRA smoke run (Qwen2.5-0.5B-Instruct, synthetic starter data)

**Status: NOT EXECUTED by its author.** This notebook has never been run. Its first real run is the first test of it.

What it does, once you run it on Kaggle:

1. Clones this repository from the published branch (`main`; the exact commit it got is recorded in `status.json`).
2. Installs the pinned Transformers, PEFT and Hub packages.
3. Runs `fabric/training/smoke.py`, which checks each stage on its own:
   `environment`, `model_verification`, `dataset`, `training`, `reload`, `inference`, `quality`, `persistence`.

Each stage is reported as **PASSED**, **FAILED** or **NOT RUN**. A stage that did not run is never shown as a pass.

**What this is not.** It is not a Klyvaron model and not a quality claim. The data is **synthetic**: it is generated by fixed rules, with answers computed by Python. It tests the pipeline, not the model's usefulness.

**Kaggle (primary).** Settings: Accelerator = GPU; **Internet = On** (needed for git, pip and the Hugging Face hub — it is off by default, and with it off the first cell stops with "Could not resolve host: github.com"). Then **Save Version > Save & Run All**. Kaggle keeps `/kaggle/working` as the output of a *committed* version. Interactive session files are not kept unless you commit.

**Colab (fallback).** Runtime > Change runtime type > GPU. Files stay on the session disk and are lost when the runtime ends. Download `status.json` and the adapter before you close it. No Drive mount is used.

**Persistence check.** After a committed run, copy the notebook, add that version's output as an Input, set `PERSISTENCE_ADAPTER` to its artifact directory under `/kaggle/input/`, and run again. The `persistence` stage stays NOT RUN until that read-back succeeds.

In [ ]:
# Settings. Edit these only if you know why.
import os

if os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
    WORK = "/kaggle/working/aetheris-smoke"
elif os.path.exists("/content"):
    PLATFORM = "colab"
    WORK = "/content/aetheris-smoke"
else:
    raise SystemExit("Run this notebook on Kaggle or Colab with a GPU accelerator.")

REPO_URL = "https://github.com/rajaram-2005/ragam.git"
REPO_REF = "main"                   # the published branch. The exact commit used is recorded in status.json.
                                    # A session branch (arena/...) is deleted when its session ends, which
                                    # would break `git clone` for every later run of this notebook.
REPO_DIR = "/tmp/ragam"             # scratch clone; not in the saved output
JOB_FILE = "services/fabric/remote/jobs/klyvaron-chat-lora-experiment.json"
PERSISTENCE_ADAPTER = None          # e.g. "/kaggle/input/<earlier-output>/.../run/artifact" to read back
print("platform:", PLATFORM, "| output:", WORK)

In [ ]:
# Clone the repository and install the pinned packages. Any failure stops the notebook here.
import subprocess, sys


def run(cmd, what):
    """Run a command, and on failure print what it said rather than raising a bare exit code."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode:
        message = ((result.stderr or "") + (result.stdout or "")).strip()
        print(message[-2000:], file=sys.stderr)
        raise SystemExit(f"{what} failed (exit {result.returncode}). The output above says why.")
    return result


if os.path.isdir(REPO_DIR):
    run(["rm", "-rf", REPO_DIR], "clearing the scratch clone")

clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
if clone.returncode:
    # Two failures account for almost every run of this cell that stops here. Say which one it
    # is and what to do, instead of an exit code that names neither.
    message = (clone.stderr or "") + (clone.stdout or "")
    print(message.strip()[-2000:], file=sys.stderr)
    if "Could not resolve host" in message or "unable to access" in message or "Could not connect" in message:
        raise SystemExit(
            "git could not reach github.com. This is almost always the notebook's Internet switch, "
            "which is off by default: on Kaggle open the Settings panel, set Internet to On, then "
            "Save Version > Save & Run All again. On Colab, reconnect the runtime. "
            "Nothing has been trained and nothing was written.")
    if "Remote branch" in message and "not found" in message:
        raise SystemExit(
            f"The branch {REPO_REF!r} does not exist any more. This copy of the notebook was saved "
            "before that branch was deleted, which happens when a working session ends. Re-import "
            "the notebook from the repository, or set REPO_REF = \"main\" in the cell above and run "
            "this cell again. "
            "Nothing has been trained and nothing was written.")
    raise SystemExit(f"git clone failed (exit {clone.returncode}); git said:\n{message.strip()[-1200:]}")

REPO_COMMIT = run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], "reading the cloned commit").stdout.strip()
print("repository commit:", REPO_COMMIT)

run([sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.19.0", "peft==0.21.2", "accelerate==1.15.0",
     "huggingface_hub==2.2.0", "safetensors==0.8.0"], "installing the pinned packages")
sys.path.insert(0, os.path.join(REPO_DIR, "services", "fabric"))

In [ ]:
# Run every stage and print the ledger. Nothing here is a quality claim.
import json
from fabric.training.hf_peft import HFPeftBackend
from fabric.training.jobspec import load_job
from fabric.training.smoke import STAGES, run_smoke

job = load_job(os.path.join(REPO_DIR, JOB_FILE))   # refuses placeholders and any broken rule
ledger = run_smoke(
    job,
    HFPeftBackend(work_dir=WORK),
    out_dir=WORK,
    platform=PLATFORM,
    persistence_adapter=PERSISTENCE_ADAPTER,
    context={"repoCommit": REPO_COMMIT, "repoRef": REPO_REF, "jobFile": JOB_FILE},
)
print("VERDICT:", ledger.verdict)
for name in STAGES:
    print(f"  {name:<20} {ledger.stages[name]['status']}")
for note in ledger.notes:
    print("note:", note)
print("status file:", os.path.join(WORK, "status.json"))

In [ ]:
# Show what the adapter is and what each core stage measured.
status = json.load(open(os.path.join(WORK, "status.json")))
print(json.dumps({name: status["stages"][name] for name in ("environment", "training", "quality", "inference")},
                 indent=2, default=str)[:4000])